# Семинар 09. HTTP и JSON


## Цели

После семинара вы сможете:

- разбирать URL, HTTP-запрос и HTTP-ответ;
- интерпретировать коды состояния и заголовок `Content-Type`;
- выбирать HTTP-метод с учётом безопасности и идемпотентности;
- сериализовать JSON и надёжно обрабатывать сетевые ответы.

> **Формат:** справочный материал для индивидуального проекта. Отдельного задания по семинару нет.

## Перед началом

Установите `requests`. Для сетевых примеров требуется подключение к интернету.


## Полезные ссылки

- [RFC 9110: HTTP Semantics](https://www.rfc-editor.org/rfc/rfc9110.html)
- [RFC 8259: JSON](https://www.rfc-editor.org/rfc/rfc8259.html)
- [Requests: Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/)

## Протокол HTTP

HTTP — прикладной протокол обмена сообщениями по модели «запрос — ответ». Клиент указывает методом, что хочет сделать, и адресует запрос ресурсу. Сервер обрабатывает запрос и возвращает код состояния, заголовки и, возможно, тело ответа. Ресурс не обязан быть файлом: это может быть пользователь, заказ, результат вычисления или коллекция объектов.

URL выполняет ту же основную работу, что адрес на отправлении: сообщает, куда направить запрос и к какому объекту внутри сервиса обратиться. Но URL содержит несколько частей с разным назначением. Разберём `https://api.example.com/v1/search?text=python&limit=10#results`:

| Часть | Значение | Назначение |
|---|---|---|
| схема | `https` | HTTP через защищённое TLS-соединение |
| хост | `api.example.com` | имя сервера |
| путь | `/v1/search` | идентификатор ресурса на сервере |
| query string | `text=python&limit=10` | параметры запроса |
| fragment | `results` | локальная часть ссылки; браузер не отправляет её серверу в HTTP-запросе |

HTTPS шифрует трафик и позволяет проверить сервер по сертификату. Это защищает данные от чтения и незаметного изменения по пути, но не гарантирует, что сам API работает правильно или что его данные можно использовать без проверки.

HTTP-запрос состоит из следующих частей:

- метод, например `GET` или `POST`;
- целевой путь и параметры;
- заголовки с метаданными, например `Accept`, `Content-Type` и `Authorization`;
- необязательное содержимое запроса.

Каждый HTTP-запрос сам по себе рассматривается отдельно от предыдущих. Если приложению нужна пользовательская сессия, клиент передаёт её идентификатор в cookie или заголовке, а сервер по этому идентификатору находит сохранённое состояние. Связность запросов обеспечивает приложение, а не скрытая память самого HTTP.

В Python есть встроенный пакет `urllib`, но в семинаре используется сторонняя синхронная библиотека `requests`. Пока `requests` ждёт сеть, вызывающий поток занят. Поэтому прямой вызов в асинхронном приложении заблокирует event loop; там нужен асинхронный HTTP-клиент или явный перенос синхронного вызова в отдельный поток.


In [ ]:
import requests


response = None
try:
    response = requests.get(
        "https://example.com",
        params={"source": "seminar"},
        headers={"Accept": "text/html"},
        timeout=(3.05, 10),  # тайм-ауты подключения и чтения
    )
    response.raise_for_status()
    print("Final URL:", response.url)
    print("Status:", response.status_code)
    print("Content-Type:", response.headers.get("Content-Type"))
except requests.RequestException as error:
    print(f"Request failed: {error}")


## HTTP-ответ

Объект `Response` содержит код состояния, заголовки, итоговый URL после перенаправлений и тело ответа. Код состояния — краткая отметка о результате обработки запроса; подробности сервер может передать отдельно в теле. Трёхзначные коды делятся на пять классов:

| Класс | Значение | Примеры |
|---|---|---|
| `1xx` | промежуточная информация | запрос принят, обработка продолжается |
| `2xx` | запрос успешно обработан | `200 OK`, `201 Created`, `204 No Content` |
| `3xx` | для завершения нужны дополнительные действия | перенаправление или использование кэша |
| `4xx` | запрос не может быть выполнен в текущем виде | `400 Bad Request`, `401 Unauthorized`, `404 Not Found`, `429 Too Many Requests` |
| `5xx` | сервер не смог выполнить допустимый запрос | `500 Internal Server Error`, `503 Service Unavailable` |

`response.raise_for_status()` выбрасывает `HTTPError` для ответов 4xx и 5xx. Это не делает тело недоступным: в нём может находиться полезное описание ошибки, код прикладной причины или список неверных полей.

Способ чтения выбирают по `Content-Type` и контракту API:

- `response.text` декодирует тело как текст;
- `response.content` возвращает байты, например для изображения;
- `response.json()` разбирает JSON, но выбрасывает исключение для пустого или некорректного тела.

Успешный разбор JSON доказывает только то, что тело имеет допустимый JSON-синтаксис. Сервер может вернуть в таком формате описание ошибки вместе со статусом 4xx или 5xx. Поэтому клиент отдельно проверяет статус, тип содержимого и структуру данных.


In [ ]:
if response is not None:
    content_type = response.headers.get("Content-Type", "")
    if content_type.startswith("text/"):
        print(response.text[:500])  # выводим только начало ответа


## Формат JSON

JSON (JavaScript Object Notation) — текстовый формат обмена структурированными данными. Он описывает данные, а не объекты Python. JSON-значением может быть объект, массив, строка, число, `true`, `false` или `null`; эти значения можно вкладывать друг в друга.

| JSON | Python после `json.loads()` |
|---|---|
| object | `dict` |
| array | `list` |
| string | `str` |
| number без точки и экспоненты | `int` |
| number с точкой или экспонентой | `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

Имена полей объекта и строки записываются в двойных кавычках. Стандартный JSON не допускает комментарии и запятую после последнего элемента. Числа `NaN` и `Infinity` также не входят в стандарт JSON, хотя некоторые библиотеки принимают их как расширение.

Встроенный модуль `json` преобразует объекты Python в JSON-строку через `dumps()` и разбирает строку через `loads()`. Функции `dump()` и `load()` выполняют те же операции непосредственно с файловым объектом. После разбора всё равно нужно проверить типы, обязательные поля и допустимые значения: синтаксически корректный JSON может не соответствовать контракту API.


In [ ]:
import json


data = {
    "name": "Иван",
    "age": 30,
    "skills": ["Python", "HTTP"],
    "active": True,
    "city": None,
}

dumped = json.dumps(data, ensure_ascii=False, indent=2)
print(dumped)

unpacked = json.loads(dumped)
assert unpacked == data


In [ ]:
import requests


# Метод json() десериализует тело ответа в объект Python.
try:
    response = requests.get(
        "https://jsonplaceholder.typicode.com/todos/1",
        headers={"Accept": "application/json"},
        timeout=(3.05, 10),
    )
    response.raise_for_status()
    media_type = response.headers.get("Content-Type", "").split(";", 1)[0]
    if media_type != "application/json" and not media_type.endswith("+json"):
        raise ValueError(f"Unexpected Content-Type: {media_type or 'missing'}")
    result = response.json()

    if not isinstance(result, dict):
        raise ValueError("Expected a JSON object")
    required_fields = {"userId", "id", "title", "completed"}
    missing_fields = required_fields - result.keys()
    if missing_fields:
        raise ValueError(f"Missing fields: {sorted(missing_fields)}")

    print(result["title"], "completed =", result["completed"])
except (requests.RequestException, ValueError) as error:
    print(f"Could not load JSON: {error}")


## HTTP-методы и их свойства

Метод сообщает серверу намерение клиента. От выбранного метода зависят допустимость автоматического повтора, работа кэшей и поведение промежуточных узлов. Важны три свойства:

- **Безопасность:** клиент не просит изменить состояние ресурса. Служебные побочные эффекты вроде записи в журнал допустимы.
- **Идемпотентность:** ожидаемый эффект нескольких одинаковых запросов на сервере совпадает с эффектом одного запроса. Это похоже на команду термостату «установить 20 градусов»: повтор той же команды не меняет итоговое состояние. Команда «прибавить один градус» устроена иначе — каждый повтор даёт новый эффект. HTTP-ответы на повторные запросы при этом могут различаться: первый `DELETE` вернёт 204, а следующий — 404.
- **Кэшируемость:** ответ разрешено сохранить и переиспользовать при выполнении условий протокола и заголовков кэширования.

| Метод | Типичное назначение | Безопасный | Идемпотентный |
|---|---|:---:|:---:|
| `GET` | получить представление ресурса | да | да |
| `HEAD` | получить те же заголовки, что для `GET`, без тела ответа | да | да |
| `POST` | передать данные на обработку, часто создать подчинённый ресурс | нет | нет в общем случае |
| `PUT` | создать или полностью заменить ресурс по известному адресу | нет | да |
| `PATCH` | частично изменить ресурс | нет | не гарантируется |
| `DELETE` | удалить ресурс | нет | да |

`GET` нельзя использовать для изменения состояния, которого просит клиент: браузер, прокси или поисковый робот может заранее загрузить, повторить или закэшировать такой запрос. Результатом могут стать действия, которых пользователь не совершал явно.

Неидемпотентный запрос нельзя автоматически повторять, если неизвестно, успел ли сервер применить первую попытку. Соединение могло оборваться уже после обработки запроса, но до получения ответа клиентом. Для критичных операций API может принимать ключ идемпотентности и по нему распознавать повтор одной операции.

В `requests` query-параметры передают через `params`, а JSON-тело — через `json`. Во втором случае библиотека сериализует объект и устанавливает подходящий `Content-Type`:

```python
payload = {"title": "Новая запись", "published": False}
response = requests.post(
    "https://api.example.com/posts",
    json=payload,
    timeout=(3.05, 10),
)
response.raise_for_status()
```

## Надёжность и безопасность клиента

- Всегда задавайте тайм-ауты подключения и чтения. В `requests` они не задают жёсткий предел длительности всего скачивания.
- Различайте сетевую ошибку, неожиданный статус, неверный формат тела и несоответствие данных контракту. Это разные причины отказа и часто требуют разной реакции.
- Переиспользуйте `requests.Session`, если выполняете серию запросов к одному сервису.
- Не размещайте токены в исходном коде и query string: они могут попасть в систему контроля версий, историю браузера, журналы прокси и серверов. Секрет обычно читают из окружения и передают через заголовок `Authorization` по HTTPS.
- Не записывайте в журнал пароли, токены и полные ответы с персональными данными.


## Где пригодится в индивидуальном проекте

HTTP понадобится, если проект предоставляет собственный API или обращается к внешнему сервису. Для каждого вызова заранее зафиксируйте контракт: метод, путь, параметры, тело, успешные статусы и формат ошибки. Для входящих и исходящих данных проверяйте не только наличие JSON, но и его структуру.

Сетевой слой лучше отделить от бизнес-логики. Одна часть кода отвечает за запрос, тайм-аут, статус и разбор ответа; другая принимает решение на основе уже проверенных данных. Тогда правила приложения можно тестировать без реального интернета, а внешний сервис — заменить без переписывания основной логики.


## Самопроверка

1. Какие части URL отправляются серверу, а какая часть остаётся в браузере?
2. Почему успешный вызов `response.json()` не доказывает, что запрос выполнен успешно?
3. Чем безопасность HTTP-метода отличается от идемпотентности?
4. Почему повтор одинакового `DELETE` может вернуть другой статус и всё же оставаться идемпотентным?
5. Чем параметры `params=` и `json=` в `requests` отличаются по расположению данных?
6. Какие ошибки должен обработать клиент внешнего API помимо статуса 5xx?


## Итоги

- HTTP описывает обмен запросами и ответами с ресурсами.
- Метод выражает намерение клиента, а статус сообщает результат обработки.
- Формат тела определяют заголовок `Content-Type` и контракт API.
- Корректный JSON нужно дополнительно проверять на ожидаемую структуру.
- Надёжный клиент задаёт тайм-ауты, обрабатывает ошибки и не раскрывает секреты.
